<a href="https://colab.research.google.com/github/eulaia/pibic-audiodescricao-llms/blob/main/LLaVA_1_6_Mistral_7B_4_bit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================================
# AUDIODESCRIÇÃO DE IMAGENS COM LLAVA 1.5 7B
# ==========================================================

!pip install -q transformers accelerate bitsandbytes sentencepiece

import os
import torch

from datetime import datetime
from PIL import Image

from transformers import (
    AutoProcessor,
    LlavaForConditionalGeneration,
    BitsAndBytesConfig
)

from google.colab import drive
from IPython.display import display

# ==========================================================
# DRIVE
# ==========================================================

drive.mount('/content/drive')

# ==========================================================
# CONFIGURAÇÕES
# ==========================================================

MODEL_ID = "llava-hf/llava-1.5-7b-hf"

folder_path = "/content/drive/MyDrive/imagens"

# ==========================================================
# PROMPT
# ==========================================================

prompt = """
Descreva a imagem de maneira objetiva e concisa, com foco em facilitar a interpretação
por um leitor de audiodescrição. Inclua os seguintes elementos:
Foco Principal: Identifique o principal sujeito ou objeto da imagem. Perspectiva da Foto:
Descreva a perspectiva da foto (ex: de frente, de lado, de cima, etc.) e o tipo de plano (ex: plano
geral, plano médio, close-up, etc.). Enquadramento: Explique como os elementos principais
estão posicionados e enquadrados na imagem. Plano de Fundo: Descreva o que está no fundo da
imagem e como ele contribui para a cena. Iluminação: Explique a iluminação da cena e como ela
afeta a percepção dos elementos na imagem. Contexto (se fornecido): Inclua detalhes sobre o
local e a época, se disponíveis. Preciso que a resposta seja no formato de parágrafo completo e
coerente, não faça em partes.
"""

# ==========================================================
# QUANTIZAÇÃO 4-BIT
# ==========================================================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# ==========================================================
# CARREGAMENTO DO MODELO
# ==========================================================

print("GPU disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nCarregando modelo...")

processor = AutoProcessor.from_pretrained(
    MODEL_ID
)

model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True
)

print("Modelo carregado!")

print("\nModelo carregado em 4-bit.")
print("Dispositivo principal:", next(model.parameters()).device)

# ==========================================================
# GERAÇÃO DA DESCRIÇÃO
# ==========================================================

def gerar_descricao(imagem):

    prompt_llava = f"""
USER: <image>

{prompt}

ASSISTANT:
"""

    inputs = processor(
        text=prompt_llava,
        images=imagem,
        return_tensors="pt"
    )

    device = next(model.parameters()).device

    inputs = {
        chave: valor.to(device)
        for chave, valor in inputs.items()
    }

    output = model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False,
        use_cache=True
    )

    resposta = processor.decode(
        output[0],
        skip_special_tokens=True
    )

    if "ASSISTANT:" in resposta:
        resposta = resposta.split("ASSISTANT:")[-1]

    return resposta.strip()

# ==========================================================
# PROCESSAMENTO
# ==========================================================

def main():

    data_atual = datetime.now()

    arquivos = os.listdir(folder_path)

    imagens = [
        arquivo
        for arquivo in arquivos
        if arquivo.lower().endswith(
            (
                ".png",
                ".jpg",
                ".jpeg"
            )
        )
    ]

    quantidade_imagens = len(imagens)

    print("\nRELATÓRIO DE GERAÇÃO DE AUDIODESCRIÇÃO")
    print(f"Data: {data_atual.strftime('%d/%m/%Y')}")
    print(f"Horário: {data_atual.strftime('%H:%M:%S')}")
    print(f"Modelo utilizado: {MODEL_ID}")
    print(f"Quantidade de imagens: {quantidade_imagens}")

    print("=" * 100)

    for arquivo in imagens:

        image_path = os.path.join(
            folder_path,
            arquivo
        )

        print(f"\nProcessando: {arquivo}")

        imagem = Image.open(
            image_path
        )

        descricao = gerar_descricao(
            imagem
        )

        display(imagem)

        print("\nDescrição gerada pelo modelo:\n")

        print(descricao)

        print("\n" + "=" * 100)

    print("\nProcessamento finalizado.")

# ==========================================================
# EXECUÇÃO
# ==========================================================

main()